In [1]:
import pandas as pd
df=pd.read_csv("house_sale.csv")

In [ ]:
"# 1. Problemin çərçivələnməsi"

## Biznes problemi

Daşınmaz əmlakın müxtəlif xüsusiyyətlərinə əsasən evlərin satış qiymətini əvvəlcədən proqnozlaşdırmaq.

Məqsəd: evin sahəsi, otaq sayı, yerləşməsi, mərtəbəsi və digər xüsusiyyətlərindən istifadə edərək onun qiymətini proqnozlaşdıran Machine Learning modeli qurmaq.

## Target dəyişən

Target dəyişən: `price`

Modelin proqnozlaşdıracağı əsas dəyişən evin satış qiymətidir.

Bu dəyişən ədədi olduğu üçün məsələmiz:

**Supervised Learning → Regression**

problemidir.

## Uğur metrikası

Modelin əsas qiymətləndirmə metrikası:

**MAE (Mean Absolute Error)**

MAE modelin proqnozlaşdırdığı qiymətlə real qiymət arasındakı orta mütləq fərqi göstərir.

Əlavə olaraq modelin performansını qiymətləndirmək üçün:

- RMSE (Root Mean Squared Error)
- R² Score

metrikalarından istifadə ediləcək.

### Uğur kriteriyası

Modelin uğurlu olması üçün MAE və RMSE dəyərlərinin mümkün qədər aşağı, R² göstəricisinin isə mümkün qədər yüksək olması gözlənilir.

In [ ]:
df.info()

In [3]:
df.shape

(100775, 51)

In [ ]:
df.describe(include='all').T

In [ ]:
df.columns.tolist()

In [ ]:
df.dtypes.value_counts()

In [5]:
missing = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': df.isnull().mean() * 100
})

missing = missing.sort_values(
    by='Missing_Percentage',
    ascending=False
)

missing

,Missing_Count,Missing_Percentage
Binanın növü,100621,99.847184
featured,97626,96.875217
vip,92179,91.470107
Torpaq sahəsi,85204,84.548747
hour_y,85144,84.489209
İpoteka,67836,67.314314
mortgage,67836,67.314314
shop_title,28757,28.535847
shop_name,28757,28.535847
products_label,28145,27.928554


In [ ]:
print("Tam duplicate sətirlər:", df.duplicated().sum())
print("Duplicate ID-lər:", df['id_x'].duplicated().sum())

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

numeric_cols
df[numeric_cols].describe().T

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col].dropna(), bins=40, kde=True)
    plt.title(f"{col} Distribution")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
print(df['Kateqoriya'].value_counts())

In [ ]:
plt.figure(figsize=(10, 6))

sns.countplot(
    data=df,
    y='Kateqoriya',
    order=df['Kateqoriya'].value_counts().index
)

plt.title("Property Category Distribution")
plt.xlabel("Count")
plt.ylabel("Category")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df,
    x='Otaq sayı',
    y='price'
)

plt.title("Price by Number of Rooms")
plt.xlabel("Number of Rooms")
plt.ylabel("Price (AZN)")
plt.ylim(0, df['price'].quantile(0.99))

plt.show()

In [7]:
df['area_m2'] = (
    df['Sahə']
    .astype(str)
    .str.replace('m²', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)

df['area_m2'] = pd.to_numeric(
    df['area_m2'],
    errors='coerce'
)

df[['Sahə', 'area_m2']].head()

,Sahə,area_m2
0,145 m²,145.0
1,90 m²,90.0
2,60 m²,60.0
3,130 m²,130.0
4,100 m²,100.0


In [12]:
df['floor'] = (
    df['Mərtəbə']
    .astype(str)
    .str.extract(r'(\d+)\s*/')[0]
)

df['total_floors'] = (
    df['Mərtəbə']
    .astype(str)
    .str.extract(r'/\s*(\d+)')[0]
)

df['floor'] = pd.to_numeric(df['floor'], errors='coerce')
df['total_floors'] = pd.to_numeric(
    df['total_floors'],
    errors='coerce'
)

df[['Mərtəbə', 'floor', 'total_floors']].head()

,Mərtəbə,floor,total_floors
0,7 / 9,7.0,9.0
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,15 / 16,15.0,16.0


In [13]:
df['land_area_sot'] = (
    df['Torpaq sahəsi']
    .astype(str)
    .str.replace('sot', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)

df['land_area_sot'] = pd.to_numeric(
    df['land_area_sot'],
    errors='coerce'
)

df[['Torpaq sahəsi', 'land_area_sot']].head()

,Torpaq sahəsi,land_area_sot
0,NaN,NaN
1,1.3 sot,1.3
2,0.1 sot,0.1
3,NaN,NaN
4,NaN,NaN


In [15]:
print("0 və ya mənfi qiymətlər:",
      (df['price'] <= 0).sum())

print("0 və ya mənfi sahələr:",
      (df['area_m2'] <= 0).sum())

print("0 və ya mənfi otaq sayı:",
      (df['Otaq sayı'] <= 0).sum())

0 və ya mənfi qiymətlər: 0
0 və ya mənfi sahələr: 0
0 və ya mənfi otaq sayı: 0


In [17]:
clean_df = df.copy()

In [18]:
clean_df = clean_df[
    (clean_df['price'] > 0) &
    (clean_df['area_m2'] > 0) &
    (clean_df['Otaq sayı'].fillna(1) > 0)
].copy()

print("Təmizləmədən sonra:", clean_df.shape)

Təmizləmədən sonra: (95856, 55)


In [19]:
drop_cols = [
    'id_x',
    'rel_url',
    'estate_rel_url_x',
    'estate_rel_url_y',
    'estate_rel_url',
    'estate_rel_url_y',
    'img_url',
    'id_y',
    'estate_id',
    'estate_details_id_x',
    'estate_details_id_y',
    'description',
    'attributes',
    'extra_info',
    'address',
    'owner_name',
    'owner_title',
    'shop_name',
    'shop_title',
    'unit_price',
    'total_price',
    'Sahə',
    'Torpaq sahəsi',
    'Mərtəbə',
    'repair',
    'bill_of_sale',
    'currency_x',
    'currency_y'
]

drop_cols = list(set(drop_cols))

clean_df = clean_df.drop(
    columns=drop_cols,
    errors='ignore'
)

print("Yeni ölçü:", clean_df.shape)
clean_df.head()

Yeni ölçü: (95856, 28)


,datetime_scrape_x,price,location,city_when,city,day_x,hour_x,vip,featured,products_label,...,Binanın növü,Kateqoriya,Otaq sayı,Təmir,Çıxarış,İpoteka,area_m2,land_area_sot,floor,total_floors
0,2024-10-05 22:07:37.60613+00,499999.0,Səbail r.,"Bakı, dünən 23:52",bakı,05.10.2024,23:52,vipped,featured,NaN,...,NaN,Köhnə tikili,4.0,var,var,NaN,145.0,NaN,7.0,9.0
1,2024-10-05 22:07:37.60613+00,77000.0,Biləcəri q.,"Bakı, dünən 23:56",bakı,05.10.2024,23:56,NaN,NaN,NaN,...,NaN,Həyət evi/Bağ evi,4.0,var,yoxdur,NaN,90.0,1.3,NaN,NaN
2,2024-10-05 22:07:37.60613+00,92000.0,İnşaatçılar m.,"Bakı, dünən 23:55",bakı,05.10.2024,23:55,NaN,NaN,NaN,...,NaN,Həyət evi/Bağ evi,3.0,var,var,NaN,60.0,0.1,NaN,NaN
3,2024-10-05 22:07:37.60613+00,95000.0,Qaraçuxur q.,"Bakı, dünən 23:55",bakı,05.10.2024,23:55,vipped,featured,NaN,...,NaN,Obyekt,NaN,var,var,var,130.0,NaN,NaN,NaN
4,2024-10-05 22:07:37.60613+00,220000.0,Əhmədli m.,"Bakı, dünən 23:52",bakı,05.10.2024,23:52,NaN,NaN,Agentlik,...,NaN,Yeni tikili,3.0,var,var,NaN,100.0,NaN,15.0,16.0


In [21]:
missing_clean = pd.DataFrame({
    'Missing_Count': clean_df.isnull().sum(),
    'Missing_Percentage': clean_df.isnull().mean() * 100
}).sort_values(
    'Missing_Percentage',
    ascending=False
)

missing_clean

,Missing_Count,Missing_Percentage
Binanın növü,95702,99.839342
featured,92871,96.885954
vip,87825,91.621808
hour_y,81058,84.562260
land_area_sot,80285,83.755842
İpoteka,63830,66.589468
mortgage,63830,66.589468
products_label,25277,26.369763
total_floors,19860,20.718578
floor,19860,20.718578


In [22]:
high_missing = missing_clean[
    missing_clean['Missing_Percentage'] > 80
].index.tolist()

high_missing

['Binanın növü', 'featured', 'vip', 'hour_y', 'land_area_sot']

In [ ]:
clean_df = clean_df.drop(
    columns=high_missing,
    errors='ignore'
)

print("Təmiz dataset ölçüsü:", clean_df.shape)

In [23]:
print("Final dataset ölçüsü:", clean_df.shape)
print("Missing values:", clean_df.isnull().sum().sum())
print("Duplicate rows:", clean_df.duplicated().sum())

Final dataset ölçüsü: (95856, 28)
Missing values: 635973
Duplicate rows: 0


In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import pandas as pd
import numpy as np

# X və y
X = clean_df.drop(columns=['price'])
y = clean_df['price']

# Numeric və categorical sütunlar
numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(include='object').columns

# Preprocessing
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# 3 model
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    )
}

# Eyni CV
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Eyni metriklər
scoring = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error',
    'R2': 'r2'
}

results = []

for name, model in models.items():

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        'Model': name,
        'MAE': -scores['test_MAE'].mean(),
        'RMSE': -scores['test_RMSE'].mean(),
        'R2': scores['test_R2'].mean()
    })

# Nəticə
results_df = pd.DataFrame(results)

results_df

In [ ]:
from sklearn.model_selection import GridSearchCV

# Ən yaxşı modeli seçirik
best_model_name = results_df.loc[
    results_df['MAE'].idxmin(), 'Model'
]

best_model = models[best_model_name]

print("Ən yaxşı model:", best_model_name)


# Pipeline
best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', best_model)
])


# Modelə uyğun hyperparameter-lər
if best_model_name == 'Linear Regression':

    param_grid = {
        'model__fit_intercept': [True, False]
    }

elif best_model_name == 'Random Forest':

    param_grid = {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_split': [2, 5],
        'model__min_samples_leaf': [1, 2]
    }

elif best_model_name == 'Gradient Boosting':

    param_grid = {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [2, 3, 5],
        'model__min_samples_split': [2, 5]
    }


# GridSearchCV
grid_search = GridSearchCV(
    estimator=best_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

# Tuning
grid_search.fit(X, y)


# Ən yaxşı parametrlər
print("\nƏn yaxşı parametrlər:")
print(grid_search.best_params_)

print("\nƏn yaxşı CV MAE:")
print(-grid_search.best_score_)

In [ ]:
# Ayrılmış test setində yekun qiymətləndirmə

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# GridSearchCV nəticəsində seçilmiş ən yaxşı model
final_model = grid_search.best_estimator_

# Yekun model bütün training məlumatında öyrədilir
final_model.fit(X_train, y_train)

# Test setində proqnoz
y_pred = final_model.predict(X_test)

# Yekun qiymətləndirmə
test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2 = r2_score(y_test, y_pred)

print("YEKUN MODELİN TEST NƏTİCƏLƏRİ")
print("=" * 40)
print(f"MAE  : {test_mae:,.2f}")
print(f"RMSE : {test_rmse:,.2f}")
print(f"R²   : {test_r2:.4f}")

In [ ]:
import joblib

# Yekun modeli saxlayırıq
joblib.dump(final_model, 'house_price_model.pkl')

print("Model uğurla saxlanıldı: house_price_model.pkl")

In [ ]:
# Saxlanılmış modeli yenidən yükləyirik
loaded_model = joblib.load('house_price_model.pkl')

print("Model uğurla yükləndi.")

In [ ]:
# Biznes Hesabatı — Ev Qiymətlərinin Proqnozlaşdırılması

## 1. Layihənin məqsədi

Bu layihənin əsas məqsədi daşınmaz əmlakın müxtəlif xüsusiyyətlərinə əsaslanaraq evlərin satış qiymətini əvvəlcədən proqnozlaşdırmaqdır.

Mövcud daşınmaz əmlak məlumatlarından istifadə etməklə evin sahəsi, otaq sayı, yerləşməsi, mərtəbəsi və digər xüsusiyyətləri ilə satış qiyməti arasındakı əlaqə analiz edilmişdir.

Hazırlanan model daşınmaz əmlak şirkətinə yeni evlər üçün daha əsaslandırılmış qiymət təyin etməyə və bazar qiymətlərini daha yaxşı analiz etməyə kömək edə bilər.

---

## 2. Əsas biznes problemi

Daşınmaz əmlak bazarında evin düzgün qiymətləndirilməsi vacibdir. Həddindən artıq yüksək qiymət satışın gecikməsinə, həddindən artıq aşağı qiymət isə potensial gəlirin azalmasına səbəb ola bilər.

Bu səbəbdən məqsəd evin mövcud xüsusiyyətlərinə əsaslanaraq onun təxmini bazar qiymətini avtomatik müəyyən edən bir sistem yaratmaqdır.

---

## 3. Məlumatların analizi

Layihədə **100,775 daşınmaz əmlak qeydi və 51 ilkin dəyişən** üzərində analiz aparılmışdır.

Məlumatların keyfiyyətini artırmaq üçün:

* boş məlumatlar müəyyən edilmiş və uyğun üsullarla doldurulmuş;
* duplicate məlumatlar yoxlanılmış;
* səhv və məntiqsiz qiymətlər analiz edilmiş;
* mətn formatında olan sahə və mərtəbə məlumatları istifadəyə yararlı formata çevrilmiş;
* model üçün əhəmiyyətsiz və ya nəticəyə birbaşa təsir göstərə biləcək bəzi məlumatlar çıxarılmışdır.

EDA nəticəsində evin sahəsi, otaq sayı, yerləşməsi və digər xüsusiyyətlərinin qiymətlə əlaqəsi araşdırılmışdır.

---

## 4. Model yanaşması

Qiymət proqnozlaşdırılması üçün üç fərqli Machine Learning modeli müqayisə edilmişdir:

* Linear Regression
* Random Forest
* Gradient Boosting

Bütün modellər eyni qiymətləndirmə şəraitində müqayisə edilmişdir. Daha sonra ən yaxşı nəticə göstərən model üçün parametrlərin optimallaşdırılması aparılmışdır.

Modelin performansını qiymətləndirmək üçün əsasən **MAE, RMSE və R²** göstəricilərindən istifadə edilmişdir.

---

## 5. Əsas nəticə

Model müqayisəsi və optimallaşdırmadan sonra ən yaxşı nəticə göstərən model yekun model kimi seçilmişdir.

Yekun model ayrılmış test məlumatları üzərində yalnız bir dəfə qiymətləndirilmişdir.

**Test nəticələri:**

* **MAE:** `[buraya nəticəni yaz]`
* **RMSE:** `[buraya nəticəni yaz]`
* **R²:** `[buraya nəticəni yaz]`

MAE göstəricisi modelin ev qiymətlərini orta hesabla nə qədər səhvlə proqnozlaşdırdığını göstərir. R² göstəricisinin yüksək olması modelin qiymətlərdəki dəyişiklikləri daha yaxşı izah etdiyini göstərir.

---

## 6. Biznes üçün faydası

Hazırlanan model daşınmaz əmlak şirkətinə aşağıdakı istiqamətlərdə dəstək verə bilər:

* Yeni əmlaklar üçün ilkin qiymət təklifi hazırlamaq;
* Bazar qiymətindən ciddi fərqlənən elanları müəyyən etmək;
* Qiymətləndirmə prosesini sürətləndirmək;
* Satış strategiyasının hazırlanmasına dəstək olmaq;
* Böyük həcmdə əmlakı avtomatik qiymətləndirmək;
* Analitik qərarların qəbulunu məlumatlara əsaslandırmaq.

---

## 7. Tövsiyə

Model biznes prosesində ilkin qiymətləndirmə və qərar dəstəyi vasitəsi kimi istifadə edilə bilər. Bununla belə, yekun satış qiyməti müəyyən edilərkən bazardakı cari vəziyyət, əmlakın real vəziyyəti və ekspert qiymətləndirməsi də nəzərə alınmalıdır.

Növbəti mərhələdə modelin real biznes sisteminə inteqrasiya edilməsi, yeni məlumatlarla mütəmadi yenilənməsi və zaman keçdikcə performansının izlənilməsi tövsiyə olunur.

### Yekun

Layihə nəticəsində daşınmaz əmlak məlumatlarından istifadə edərək evlərin qiymətini proqnozlaşdıran və biznes qərarlarının qəbuluna dəstək ola biləcək Machine Learning həlli hazırlanmışdır.
